In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27133, done.
remote: Total 27133 (delta 0), reused 0 (delta 0), pack-reused 27133 (from 1)
Receiving objects: 100% (27133/27133), 13.01 MiB | 22.70 MiB/s, done.
Resolving deltas: 100% (19493/19493), done.


In [2]:
!pwd

/content


In [3]:
%cd /content/LLaMA-Factory

/content/LLaMA-Factory


In [4]:
!pwd

/content/LLaMA-Factory


In [5]:
!ls

assets	      data    examples	MANIFEST.in	README_zh.md  src
CITATION.cff  docker  LICENSE	pyproject.toml	requirements  tests
CLAUDE.md     docs    Makefile	README.md	scripts       tests_v1


In [9]:
!pip install -e .

Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 114.0 MB/s eta

In [10]:
!pip install bitsandbytes>=0.39.0

In [12]:
!ls /content/LLaMA-Factory/src

api.py	llamafactory  train.py	webui.py


In [14]:
# import os
# os.environ["GRADIO_SHARE"] = "1" # Web UI will be loaded using GRADIO UI

In [ ]:
!pip install accelerate transformers peft

In [ ]:
!hf auth login

In [ ]:
!python /content/LLaMA-Factory/src/webui.py

### Method 2

In [ ]:
from llamafactory.webui.interface import create_ui
ui = create_ui()
ui.launch(share=True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Base model (same as you used in training)
base_model = "google/gemma-1.1-2b-it"

# Path to your trained LoRA adapter
lora_path = "/content/LLaMA-Factory/saves/Gemma-1.1-2B-Instruct/lora/train_2025-12-11-17-40-51"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

# Apply LoRA on top of base model
model = PeftModel.from_pretrained(model, lora_path)

In [ ]:
# Load tokenizer

# Enable evaluation mode
model.eval()

# --- Inference prompt ---
prompt = "Can you tell what ls -l would display?"

# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

# Decode and print
print(tokenizer.decode(outputs[0], skip_special_tokens=True))